In [18]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import h5py
from cubio import write_envi, cubedata_from_json_file, CubeData, cube_from_numpy
import numpy as np

from pym3tools.rdn2rfl import pipeline_steps as r2r
from pym3tools.rdn2rfl import M3Level2Pipeline
from pym3tools.rdn2rfl import Step
from pym3tools.save_models import cache_to_cubio, PipelineCache
from pym3tools.save_models.pipeline_cache_schema import Dataset
from pym3tools.save_models.attribute_models import StandardDatasetAttrs
from pyresample.geometry import AreaDefinition
from pyresample.kd_tree import resample_nearest
from pym3tools.rdn2rfl.pipeline_services.georeference import PixelResolution
from pym3tools.constants import MOON_RADIUS
from pym3tools.geo_ops import mosaic_arrays
import matplotlib.pyplot as plt

plt.switch_backend("qtagg")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
catalog = Path("D:/moon_data/m3/Gruithuisen_Region/M3G20090208T160125/")
gcps = "D:/moon_data/m3/Gruithuisen_Region/gcps_files/M3G20090208T160125.gcps"
cache = catalog / "pipeline_cache.hdf5"
slope = "D:/moon_data/derived_products/gruithuisen_region/Gruit_Slope.json"
aspect = "D:/moon_data/derived_products/gruithuisen_region/Gruit_Aspect.json"

In [ ]:
print(f"Running Pipeline for ID: {catalog.name}")
steps: list[Step] = [
    r2r.Georeference(gcps_fp=gcps, custom_aspect=aspect, custom_slope=slope),
    r2r.SolarRemoval(),
    r2r.StatisticalPolish(),
    r2r.ThermalCorrection("Clark_Modified"),
    r2r.PhotometricCorrection("Lommel-Seeliger"),
]

pipeline = M3Level2Pipeline(
    steps,
    catalog,
    cache,
    overwrite_cache=True,
)

_ = pipeline.run()

In [ ]:
def save_suite(cache_fp: Path | str, save_dir: Path | str):
    if not Path(save_dir).exists():
        Path(save_dir).mkdir()
    with h5py.File(str(cache_fp)) as f:
        c = PipelineCache(f)
        m3id = Path(cache_fp).parent.name
        save_dict: dict[str, Dataset[StandardDatasetAttrs]] = {
            f"{m3id}_georef": c.georeferenced.cube,
            f"{m3id}_obs": c.georeferenced.obs,
            f"{m3id}_photo": c.photometric_corrected.photometry_backplane,
            f"{m3id}_rfl": c.photometric_corrected.cube
        }

        for k, v in save_dict.items():
            ctxt, cub = cache_to_cubio(v, k)
            write_envi(ctxt, cub, "BIL", Path(save_dir) / "_")

In [ ]:
cache_list: list[Path] = []
for r, d, f in Path("D:/moon_data/m3/Gruithuisen_Region/").walk():
    if r.name == "targeted_mixing_endmembers":
        continue
    for i in f:
        if Path(i).suffix == ".hdf5":
            cache_fp = Path(r, i)
            cache_list.append(cache_fp)

for i in cache_list:
    save_suite(i, i.parent / "products")

Lazy Loading from: D:\moon_data\m3\Gruithuisen_Region\M3G20090208T160125\products\M3G20090208T160125_rfl.bil
